# TE-PAI vs. Trotter — snapshot estimation with error bars

We evolve a **7-qubit Heisenberg spin chain** from a Néel state and estimate
$\langle Z_0(t)\rangle$ two ways, from finite measurement **snapshots**:

1. **Exact** — statevector expectation of the first-order Trotter circuit (reference line).
2. **Trotter** — measure the Trotter circuit with `Ns` shots.
3. **TE-PAI** — sample `M` shallow random circuits (one snapshot each), signed-weighted by the overhead.

Both samplers are unbiased for the Trotter-evolved value, so they track the exact curve within error bars; TE-PAI trades larger variance for much shallower circuits.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pai_shadow.hamil import Heisenberg_Hamil
from pai_shadow.backend import get_backend
from pai_shadow.trotter import trotter_circuit
from pai_shadow.te_pai import TEPAI

np.random.seed(0)

In [ ]:
# 7-qubit Heisenberg spin chain
n_qubits = 7
H = Heisenberg_Hamil(n_qubits, 1.0, 1.0, 1.0)
backend = get_backend("qulacs")  # or "qiskit"

# Neel initial state |0101010>
neel = [q % 2 for q in range(n_qubits)]
idx = sum(b << q for q, b in enumerate(neel))
psi0 = np.zeros(1 << n_qubits, dtype=complex)
psi0[idx] = 1.0

# Observable: <Z_0>
observable = "Z" + "I" * (n_qubits - 1)

# parameters (n_steps large enough that 2|coef|*dt <= delta for TE-PAI)
n_steps = 40
delta   = np.pi / 32
M       = 3000   # TE-PAI circuits (1 snapshot each)
Ns      = 3000   # Trotter measurement shots
times   = np.linspace(0.0, 1.5, 11)

def z0(bitstring):
    # qubit 0 is the right-most char; Z eigenvalue +1 for '0', -1 for '1'
    return 1 - 2 * int(bitstring[-1])

In [ ]:
exact, trot_mean, trot_err, tepai_mean, tepai_err = [], [], [], [], []
for t in times:
    circ = trotter_circuit(H, t, n_steps, init_state=psi0)
    # exact reference
    exact.append(backend.expectation(circ, observable))
    # Trotter with finite shots
    vals = np.array([z0(b) for b in backend.sample(circ, Ns)])
    trot_mean.append(vals.mean()); trot_err.append(vals.std() / np.sqrt(Ns))
    # TE-PAI: M random circuits, one snapshot each, signed-weighted
    tp = TEPAI(H, delta, t, n_steps, init_state=psi0)
    circuits, weights = tp.sample(M)
    w = np.array([wt * z0(backend.sample(c, 1)[0]) for c, wt in zip(circuits, weights)])
    tepai_mean.append(w.mean()); tepai_err.append(w.std() / np.sqrt(M))
    print(f"t={t:.2f}  exact={exact[-1]:+.3f}  overhead={tp.overhead:.2f}")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(times, exact, "k-", lw=2, label="exact (statevector)")
plt.errorbar(times, trot_mean, yerr=trot_err, fmt="o", capsize=3,
             color="tab:blue", label=f"Trotter, {Ns} shots")
plt.errorbar(times, tepai_mean, yerr=tepai_err, fmt="s", capsize=3,
             color="tab:red", label=f"TE-PAI, {M} snapshots")
plt.axhline(0, color="gray", lw=0.5)
plt.xlabel("time $t$"); plt.ylabel(r"$\langle Z_0(t)\rangle$")
plt.title("7-qubit Heisenberg chain: Trotter vs TE-PAI (Neel initial state)")
plt.legend(); plt.tight_layout(); plt.show()

## What to look for

- Both samplers' points sit on the **exact** curve within their error bars (unbiased).
- **TE-PAI error bars are larger** at equal budget — the cost of the quasiprobability `overhead` $\gamma$ — but each TE-PAI circuit is much **shallower** than the Trotter circuit, which is what buys noise robustness on hardware.
- Try lowering `delta` or raising `n_steps`; note `overhead` and the TE-PAI bars change accordingly.